# 🛡️ VoiceShield v3: Frontier Gemini & Multimodal Speech Adaptation
### Automated Fine-Tuning Pipeline for Google Gemini, Modern Generative Audio & Telephonic Distortion
---
**Hardware Target:** Google Colab Free Tier (T4 GPU)
**Total Execution Time:** ~2 minutes
**Key Objective:** Adapt the existing `v3-hardened` model to detect out-of-distribution conversational AI voices (Google Gemini 1.5/2.0, OpenAI GPT-4o voice, ElevenLabs) across clean and distorted telephonic channels without catastrophic forgetting.


## Step 1: Install Dependencies & Verify GPU Runtime
> In the top menu, verify **Runtime** -> **Change runtime type** -> **T4 GPU** is selected.

In [ ]:
!pip install -q transformers accelerate datasets soundfile edge-tts librosa scipy evaluate huggingface_hub

import torch
print('=' * 60)
print('PyTorch Version:', torch.__version__)
print('CUDA GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Active GPU Device:', torch.cuda.get_device_name(0))
    print(f'Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB')
else:
    print('WARNING: GPU not detected. Select Runtime -> Change runtime type -> T4 GPU.')
print('=' * 60)

## Step 2: Define Acoustic Distortion & Augmentation Operators
Simulates telephonic channels (300-3400Hz bandpass + non-linear saturation), physical room echoes (RIR), and ambient microphone noise.

In [ ]:
import numpy as np
import scipy.signal

def apply_room_reverb(audio: np.ndarray, sr: int = 16000, decay_sec: float = 0.22) -> np.ndarray:
    """Simulates physical loudspeaker-to-microphone room reflections (RIR)."""
    t = np.linspace(0, decay_sec, int(sr * decay_sec))
    rir = np.exp(-t * 14) * np.random.randn(len(t))
    rir[0] = 1.0
    rir /= (np.sum(np.abs(rir)) + 1e-9)
    rev = scipy.signal.convolve(audio, rir, mode='same')
    out = 0.75 * audio + 0.25 * rev
    return np.clip(np.nan_to_num(out), -1.0, 1.0).astype(np.float32)

def apply_distortion_and_bandpass(audio: np.ndarray, sr: int = 16000) -> np.ndarray:
    """Simulates telephone bandpass filter (300-3400Hz) and clipping saturation."""
    sos = scipy.signal.butter(4, [300, 3400], btype='bandpass', fs=sr, output='sos')
    filtered = scipy.signal.sosfilt(sos, audio)
    dist = np.tanh(filtered * 1.5)
    max_val = np.max(np.abs(dist)) + 1e-6
    out = dist / max_val
    return np.clip(np.nan_to_num(out), -1.0, 1.0).astype(np.float32)

def apply_ambient_noise(audio: np.ndarray, snr_db: float = 22.0) -> np.ndarray:
    """Injects calibrated ambient room air noise floor."""
    noise = np.random.randn(len(audio))
    sig_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)
    factor = np.sqrt(sig_power / (10 ** (snr_db / 10) * noise_power + 1e-9))
    out = audio + factor * noise
    return np.clip(np.nan_to_num(out), -1.0, 1.0).astype(np.float32)

print('[+] Acoustic simulation operators defined.')

## Step 3: Ingest Custom Gemini Audio & Generate Frontier Audio Samples
* Upload your own `.wav` files (e.g. Gemini mobile/web recordings) into the `incoming_gemini/` folder.
* Automatically synthesizes conversational English and Hindi frontier samples and expands them using augmentations.

In [ ]:
import os, glob, asyncio, soundfile as sf, librosa, edge_tts
from pathlib import Path

Path('incoming_gemini').mkdir(exist_ok=True)
Path('dataset/gemini_frontier').mkdir(parents=True, exist_ok=True)
Path('dataset/human_anchors').mkdir(parents=True, exist_ok=True)

# 1. Synthesize conversational English & Hindi speech
async def synthesize_frontier_voices():
    prompts = [
        ('Master tip: Keep the conversation direct and to the point, that is the true sigma trait.', 'en-US-AndrewMultilingualNeural', 'en_sigma_andrew'),
        ('I will immediately dispatch the encrypted security verification code to your mobile phone.', 'en-US-AvaMultilingualNeural', 'en_bank_ava'),
        ('Recent advances in neural audio synthesis can mimic organic human speech with high fidelity.', 'en-US-BrianMultilingualNeural', 'en_neural_brian'),
        ('मास्टर टिप: बात को हमेशा सीधा और टू द पॉइंट रखना भी एक सिग्मा ट्रेट है मित्र।', 'hi-IN-MadhurNeural', 'hi_sigma_madhur'),
        ('यह आपके बैंक खाते की सुरक्षा से संबंधित आवश्यक सत्यापन संदेश है।', 'hi-IN-SwaraNeural', 'hi_bank_swara')
    ]
    for text, voice, name in prompts:
        mp3 = f'tmp_{name}.mp3'
        comm = edge_tts.Communicate(text, voice)
        await comm.save(mp3)
        y, _ = librosa.load(mp3, sr=16000)
        if os.path.exists(mp3): os.remove(mp3)
        y4 = y[:16000 * 4]
        sf.write(f'dataset/gemini_frontier/{name}_clean.wav', y4, 16000)
        sf.write(f'dataset/gemini_frontier/{name}_dist.wav', apply_distortion_and_bandpass(y4), 16000)
        sf.write(f'dataset/gemini_frontier/{name}_rev.wav', apply_room_reverb(y4), 16000)

await synthesize_frontier_voices()

# 2. Ingest any user uploaded audio files from incoming_gemini/
custom_incoming = glob.glob('incoming_gemini/*.wav') + glob.glob('incoming_gemini/*.mp3')
for idx, f in enumerate(custom_incoming):
    y, sr = sf.read(f)
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
    sf.write(f'dataset/gemini_frontier/custom_gemini_{idx}_raw.wav', y[:16000 * 4], 16000)
    sf.write(f'dataset/gemini_frontier/custom_gemini_{idx}_dist.wav', apply_distortion_and_bandpass(y[:16000 * 4]), 16000)
    sf.write(f'dataset/gemini_frontier/custom_gemini_{idx}_rev.wav', apply_room_reverb(y[:16000 * 4]), 16000)

frontier_clips = glob.glob('dataset/gemini_frontier/*.wav')
print(f'[+] Prepared {len(frontier_clips)} frontier synthetic speech clips across clean, phone-distorted, and reverb channels.')

## Step 4: Stream Real Human Anchors (Google FLEURS) to Prevent Catastrophic Forgetting
Interleaves authentic human voices (English and Hindi) to maintain a strict 50/50 balance.

In [ ]:
from datasets import load_dataset

print('Streaming authentic human speech anchors from Google FLEURS...')
human_target = len(frontier_clips)
human_count = 0

for lang in ['en_us', 'hi_in']:
    ds = load_dataset('google/fleurs', lang, split='train', streaming=True)
    for item in ds:
        if human_count >= human_target: break
        y = np.array(item['audio']['array'], dtype=np.float32)
        sr = item['audio']['sampling_rate']
        if sr != 16000: y = librosa.resample(y, orig_sr=sr, target_sr=16000)
        if len(y) >= 16000 * 3.0:
            sf.write(f'dataset/human_anchors/fleurs_{lang}_{human_count}.wav', y[:16000 * 4], 16000)
            human_count += 1

human_clips = glob.glob('dataset/human_anchors/*.wav')
print(f'[+] Balanced Dataset: {len(human_clips)} Real Human vs. {len(frontier_clips)} AI Synthetic (1:1 Ratio).')

## Step 5: Load Base v3-hardened Model & Freeze Lower Layers
Freezes all 7 convolutional feature encoder layers and bottom 20 transformer layers so previously learned representations remain 100% intact.

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

base_checkpoint = 'ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened'
print(f'Loading base checkpoint: {base_checkpoint}...')

feature_extractor = AutoFeatureExtractor.from_pretrained(base_checkpoint)
model = AutoModelForAudioClassification.from_pretrained(base_checkpoint)

# 1. Freeze lower convolutional encoder layers
model.freeze_feature_encoder()

# 2. Freeze bottom 20 transformer layers (fine-tune top 4 layers + classification head)
if hasattr(model, 'wav2vec2') and hasattr(model.wav2vec2, 'encoder'):
    for layer in model.wav2vec2.encoder.layers[:20]:
        for param in layer.parameters():
            param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'[+] Feature encoder frozen. Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%).')

## Step 6: Accelerated T4 GPU Training Loop (~60-90 seconds)
Runs with `fp16=True`, batch size 16, and low learning rate ($2.5 \times 10^{-5}$) with gradient clipping.

In [ ]:
import random
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
import evaluate

class FastDataset(Dataset):
    def __init__(self, records, extractor):
        self.records = records
        self.extractor = extractor
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        path, label = self.records[idx]
        y, sr = sf.read(path)
        if y.ndim > 1: y = y.mean(axis=1)
        inputs = self.extractor(y, sampling_rate=16000, max_length=16000 * 4, truncation=True, padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

all_samples = [(f, 0) for f in human_clips] + [(f, 1) for f in frontier_clips]
random.seed(42)
random.shuffle(all_samples)

split = int(len(all_samples) * 0.85)
train_ds = FastDataset(all_samples[:split], feature_extractor)
val_ds = FastDataset(all_samples[split:], feature_extractor)

acc_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

train_args = TrainingArguments(
    output_dir='./voiceshield-v3-adapted',
    eval_strategy='epoch',
    learning_rate=2.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print('⚡ Starting T4 GPU Adaptation Training...')
trainer.train()
print('[+] Training complete!')

## Step 7: Multi-Condition Validation Benchmark
Verifies accuracy across Gemini frontier speech, authentic FLEURS human speech, and telephonic channels.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

print('=' * 75)
print('VALIDATION BENCHMARK RESULTS')
print('=' * 75)

for sample_path, expected in [(frontier_clips[0], 'AI Synthetic'), (human_clips[0], 'Real Human')]:
    y, _ = sf.read(sample_path)
    inputs = feature_extractor(y, sampling_rate=16000, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(inputs.input_values).logits.squeeze()
        probs = torch.softmax(logits / 2.4, dim=-1).cpu().tolist()
    fake_prob = probs[1]
    pred = 'AI_GENERATED' if fake_prob >= 0.50 else 'HUMAN_SPEECH'
    print(f'{Path(sample_path).name:35} | Expected: {expected:14} -> Pred: {pred} (Fake: {fake_prob:.2%})')
print('=' * 75)

## Step 8: Save & Export Checkpoint
* Save locally to download the `.zip` file into your local VoiceShield repository, or push directly to Hugging Face Hub.

In [ ]:
# Save final model locally
output_dir = './voiceshield-v3-hardened-adapted'
trainer.save_model(output_dir)
feature_extractor.save_pretrained(output_dir)

# Option A: Zip and download to local machine
!zip -r voiceshield_adapted.zip voiceshield-v3-hardened-adapted
from google.colab import files
# files.download('voiceshield_adapted.zip')  # Uncomment to trigger browser download

# Option B: Push directly to Hugging Face
# from huggingface_hub import login
# login()  # Enter your HF Write Token
# model.push_to_hub('ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened')
# feature_extractor.push_to_hub('ArmanSandhu1729/voiceshield-hindi-english-deepfake-v3-hardened')
print('[+] Model saved and ready for deployment in VoiceShield!')